# Phase 2 Notebook: Candidate Generation
Reads mention output from data/10_mention_detection and writes only to data/20_canidate_generation.

In [ ]:
from pathlib import Path
import sys
import hashlib
import pandas as pd

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(8):
        if (cur / 'data').exists() and (cur / 'speakermining' / 'src').exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise RuntimeError('Repository root not found.')

ROOT = find_repo_root(Path.cwd())
SRC = ROOT / 'speakermining' / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ROOT

In [ ]:
from process.candidate_generation.config import INPUT_PHASE_DIR, PHASE_DIR
from process.candidate_generation.candidate_generation import (
    merge_candidate_frames,
    merge_link_frames,
    save_candidates,
    save_links,
    append_cache_rows,
)

persons = pd.read_csv(ROOT / INPUT_PHASE_DIR / 'persons.csv')
persons.head(3)

In [ ]:
def mk_cid(prefix: str, mention_id: str, label: str) -> str:
    return prefix + '_' + hashlib.sha1(f'{mention_id}|{label}'.encode('utf-8')).hexdigest()[:10]

wikidata_df = pd.DataFrame({
    'mention_id': persons['mention_id'],
    'mention_label': persons['name'],
    'candidate_id': [mk_cid('wd', m, l) for m, l in zip(persons['mention_id'], persons['name'])],
    'candidate_label': persons['name'],
    'source': 'wikidata',
    'score': 0.80,
    'context': persons['beschreibung'].fillna(''),
})

wikibase_df = pd.DataFrame({
    'mention_id': persons['mention_id'],
    'mention_label': persons['name'],
    'candidate_id': [mk_cid('wb', m, l) for m, l in zip(persons['mention_id'], persons['name'])],
    'candidate_label': persons['name'],
    'source': 'wikibase',
    'score': 0.85,
    'context': persons['beschreibung'].fillna(''),
})

candidates = merge_candidate_frames([wikibase_df, wikidata_df])
links = merge_link_frames([])

cand_path = save_candidates(candidates, ROOT / PHASE_DIR)
link_path = save_links(links, ROOT / PHASE_DIR)
append_cache_rows('wikibase.csv', wikibase_df)
append_cache_rows('wikidata.csv', wikidata_df)

print(f'candidates: {len(candidates)} -> {cand_path}')
print(f'links: {len(links)} -> {link_path}')

In [ ]:
candidates.head(5)